## Data

In [1]:
import numpy as np
import os
import pandas as pd

path = os.path.join('Data', 'test.csv')
data = pd.read_csv(path)

x, y_name = data.drop('Air_Quality', axis=1), data['Air_Quality']
y_dum = pd.get_dummies(y_name)
y = np.argmax(y_dum[y_dum.columns].values, axis=1)

x

,Temperature,Humidity,PM10,NO2,SO2,CO
0,0.276596,-0.628249,0.313364,-0.203390,-0.032258,0.075949
1,-0.946809,-0.171751,-0.685714,-0.703390,-0.634409,-0.481013
2,0.063830,-0.302825,-0.228571,-0.279661,-0.505376,-0.303797
3,0.606383,0.099435,0.420276,0.728814,0.505376,1.139241
4,0.393617,0.230508,0.011060,-0.618644,0.376344,0.215190
...,...,...,...,...,...,...
995,1.287234,0.555932,1.128111,0.881356,0.688172,1.341772
996,0.712766,0.149153,-0.523502,0.135593,0.645161,0.468354
997,0.904255,-0.501695,0.022120,-0.279661,0.075269,0.202532
998,-0.893617,-0.284746,0.051613,0.398305,-0.021505,0.177215


## Getting Predictors
Build on top of three best predictors

In [6]:
from utils.Predictor import Predictor

predictors = {}

for i in range(1,4):
    name = f'Top_{i}'
    predictors[name] = Predictor(path=os.path.join('Models', name))

predictors

{'Top_1': <utils.Predictor.Predictor at 0x70c63e4cb850>,
 'Top_2': <utils.Predictor.Predictor at 0x70c5899a4f90>,
 'Top_3': <utils.Predictor.Predictor at 0x70c588ede710>}

## Friedman Test

In [19]:
from utils.stat_testing import compare_ensembles


y_true = y
ensemble_names = list(predictors.keys())
ensemble_predictions = []
for idx, p in enumerate(predictors.values()):
    pred = p.predict(x)
    ensemble_predictions.append(pred)

results = compare_ensembles(ensemble_predictions=ensemble_predictions, y_true=y_true, ensemble_names=ensemble_names)
results_df = pd.DataFrame(results)

results_df

,k,n,alpha,ensemble_scores,mean_ranks,critical_difference,friedman_stat,friedman_p,significant,pairwise
Top_1,3,1000,0.05,0.919,1.999,0.1149,0.5,0.778801,False,NaN
Top_2,3,1000,0.05,0.919,1.999,0.1149,0.5,0.778801,False,NaN
Top_3,3,1000,0.05,0.917,2.002,0.1149,0.5,0.778801,False,NaN


## Simple test for decision coverage

In [27]:
y_coverage = (ensemble_predictions[0] == ensemble_predictions[1]) & (ensemble_predictions[1] == ensemble_predictions[2]) & (ensemble_predictions[0] == ensemble_predictions[2])

pred_covrage = y_coverage.sum()/len(y_true) * 100

print(f'{pred_covrage}% of ensembles prediction are the same')

98.4% of ensembles prediction are the same
